# 🎤 Vocalido DiffSinger Training — Google Colab A100
**Resume from Step 7,000 → Target: 160,000 steps**

## 📋 ขั้นตอนก่อนรัน Notebook นี้:
1. เลือก Runtime → **A100 GPU** (Runtime → Change runtime type → A100)
2. อัปโหลดไฟล์เหล่านี้ไปยัง Google Drive ใน folder `MyDrive/vocalido_training/`:
   - `vocalido_data.zip` (zip ของ folder `vocalido/` และ `vocalido_bin/`)
   - `checkpoint_7000.zip` (zip ของ checkpoint step 7000 + config files)
3. กด **Run All** ได้เลย!

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 1: ตรวจสอบ GPU และ Mount Google Drive
# ═══════════════════════════════════════════════════════
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted')

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 3: Clone DiffSinger + Install Dependencies
# ═══════════════════════════════════════════════════════
import os

if not os.path.exists(WORK_DIR):
    print("📥 Cloning DiffSinger...")
    !git clone https://github.com/openvpi/DiffSinger.git {WORK_DIR}
else:
    print("✅ DiffSinger already cloned")

%cd {WORK_DIR}
print(f"📂 Working dir: {os.getcwd()}")

print("\n📦 Installing PyTorch (CUDA 12.1)...")
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

req_file = f"{WORK_DIR}/requirements.txt"
if os.path.exists(req_file):
    print("📦 Installing DiffSinger requirements...")
    !pip install -q -r {req_file}
else:
    print(f"⚠️  requirements.txt not found — installing core deps manually")
    !pip install -q numpy scipy librosa soundfile omegaconf einops

!pip install -q tensorboard librosa scipy matplotlib
print("✅ Dependencies installed")

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 3: Clone DiffSinger + Install Dependencies
# ═══════════════════════════════════════════════════════
import os

if not os.path.exists(WORK_DIR):
    print('📥 Cloning DiffSinger...')
    !git clone https://github.com/openvpi/DiffSinger.git {WORK_DIR}
else:
    print('✅ DiffSinger already cloned')

os.chdir(WORK_DIR)
print(f'📂 Working dir: {os.getcwd()}')

print('\n📦 Installing dependencies...')
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q -r requirements.txt
!pip install -q tensorboard librosa scipy matplotlib
print('✅ Dependencies installed')

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 4: Extract Training Data from Google Drive
# ═══════════════════════════════════════════════════════
import os, zipfile, shutil

# ── Vocoder (NSF-HiFiGAN) ───────────────────────────────
VOCODER_NAME = 'pc_nsf_hifigan_44.1k_hop512_128bin_2025.02'
VOCODER_DST  = f'{WORK_DIR}/checkpoints/{VOCODER_NAME}'

vocoder_zip = f'{DRIVE_BASE}/vocoder.zip'
if os.path.exists(vocoder_zip) and not os.path.exists(VOCODER_DST):
    print('📦 Extracting vocoder...')
    with zipfile.ZipFile(vocoder_zip) as z:
        z.extractall(f'{WORK_DIR}/checkpoints/')
    print('✅ Vocoder extracted')
elif os.path.exists(VOCODER_DST):
    print('✅ Vocoder already present')
else:
    print('⚠️  vocoder.zip not found in Drive — will try to download')
    !wget -q -O /tmp/vocoder.zip "https://huggingface.co/openvpi/vocoders/resolve/main/pc_nsf_hifigan_44.1k_hop512_128bin_2025.02.zip" 2>/dev/null || echo 'Download failed — please upload vocoder.zip manually'
    if os.path.exists('/tmp/vocoder.zip'):
        with zipfile.ZipFile('/tmp/vocoder.zip') as z:
            z.extractall(f'{WORK_DIR}/checkpoints/')
        print('✅ Vocoder downloaded and extracted')

# ── Training Data ───────────────────────────────────────
data_zip = f'{DRIVE_BASE}/vocalido_data.zip'
if os.path.exists(data_zip):
    print('\n📦 Extracting training data...')
    with zipfile.ZipFile(data_zip) as z:
        z.extractall(DATA_DIR)
    # List what was extracted
    for item in os.listdir(DATA_DIR):
        print(f'  → {item}')
    print('✅ Training data extracted')
else:
    print(f'❌ {data_zip} not found!')
    print('Please upload vocalido_data.zip to Google Drive first')
    raise FileNotFoundError('vocalido_data.zip missing from Drive')

# Detect actual paths
RAW_DATA_DIR = None
BIN_DATA_DIR = None
for item in os.listdir(DATA_DIR):
    full = os.path.join(DATA_DIR, item)
    if os.path.isdir(full):
        if 'bin' in item:
            BIN_DATA_DIR = full
        else:
            RAW_DATA_DIR = full

print(f'\n📂 Raw data : {RAW_DATA_DIR}')
print(f'📂 Bin data : {BIN_DATA_DIR}')

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 5: Restore Checkpoint (step 7,000) from Drive
# ═══════════════════════════════════════════════════════
import os, zipfile, glob

ckpt_zip = f'{DRIVE_BASE}/checkpoint_7000.zip'
existing_ckpts = glob.glob(f'{CKPT_DIR}/model_ckpt_steps_*.ckpt')

if existing_ckpts:
    latest = sorted(existing_ckpts)[-1]
    step = int(latest.split('steps_')[-1].replace('.ckpt',''))
    print(f'✅ Checkpoint already present: step {step:,}')
elif os.path.exists(ckpt_zip):
    print('📦 Extracting checkpoint from Drive...')
    with zipfile.ZipFile(ckpt_zip) as z:
        z.extractall(CKPT_DIR)
    existing = glob.glob(f'{CKPT_DIR}/model_ckpt_steps_*.ckpt')
    if existing:
        step = int(sorted(existing)[-1].split('steps_')[-1].replace('.ckpt',''))
        print(f'✅ Checkpoint restored: step {step:,}')
    else:
        print('❌ No .ckpt files found after extraction')
        raise FileNotFoundError('checkpoint_7000.zip did not contain .ckpt files')
else:
    print(f'❌ {ckpt_zip} not found in Drive!')
    raise FileNotFoundError('checkpoint_7000.zip missing from Drive')

# List checkpoint dir
print('\n📋 Checkpoint directory contents:')
for f in sorted(os.listdir(CKPT_DIR)):
    sz = os.path.getsize(os.path.join(CKPT_DIR, f)) / 1e6
    print(f'  {f}  ({sz:.1f} MB)')

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 6: Update Config Paths for Colab Environment
# ═══════════════════════════════════════════════════════
import yaml, os, shutil

config_src = f'{CKPT_DIR}/config.yaml'
config_dst = f'{WORK_DIR}/usr/configs/vocalido.yaml'

os.makedirs(os.path.dirname(config_dst), exist_ok=True)
shutil.copy(config_src, config_dst)

with open(config_dst, 'r') as f:
    cfg = yaml.safe_load(f)

# Update paths for Colab
cfg['binary_data_dir'] = BIN_DATA_DIR
cfg['max_updates'] = TARGET_STEPS

# Update raw_data_dir in datasets list
if 'datasets' in cfg:
    for ds in cfg['datasets']:
        ds['raw_data_dir'] = RAW_DATA_DIR

# A100 Optimization: increase batch size
cfg['max_batch_size'] = 32   # A100 80GB can handle 32
cfg['max_tokens'] = cfg.get('max_tokens', 40000)

# Save updated config
with open(config_dst, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False, allow_unicode=True)

# Copy helper files
for fname in ['spk_map.json', 'lang_map.json', 'dictionary-en.txt']:
    src = os.path.join(CKPT_DIR, fname)
    if os.path.exists(src):
        shutil.copy(src, f'{CKPT_DIR}/{fname}')

print('✅ Config updated for Colab:')
print(f'  binary_data_dir : {cfg["binary_data_dir"]}')
print(f'  max_updates     : {cfg["max_updates"]:,}')
print(f'  max_batch_size  : {cfg["max_batch_size"]}')
if 'datasets' in cfg:
    print(f'  raw_data_dir    : {cfg["datasets"][0]["raw_data_dir"]}')

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 7: Setup Auto-Save to Drive (ทุก 2000 steps)
# ═══════════════════════════════════════════════════════
import threading, time, glob, shutil, os

DRIVE_CKPT_BACKUP = f'{DRIVE_BASE}/checkpoints'
os.makedirs(DRIVE_CKPT_BACKUP, exist_ok=True)

_stop_autosave = threading.Event()

def autosave_worker():
    saved = set()
    while not _stop_autosave.is_set():
        time.sleep(120)  # Check every 2 minutes
        ckpts = glob.glob(f'{CKPT_DIR}/model_ckpt_steps_*.ckpt')
        for ckpt in sorted(ckpts):
            step = int(ckpt.split('steps_')[-1].replace('.ckpt',''))
            if step not in saved and step % 2000 == 0:
                dst = f'{DRIVE_CKPT_BACKUP}/model_ckpt_steps_{step}.ckpt'
                if not os.path.exists(dst):
                    print(f'\n💾 Auto-saving step {step:,} to Drive...')
                    shutil.copy2(ckpt, dst)
                    print(f'✅ Saved: {os.path.basename(dst)}')
                saved.add(step)

_autosave_thread = threading.Thread(target=autosave_worker, daemon=True)
_autosave_thread.start()
print('✅ Auto-save thread started (saves to Drive every 2000 steps)')
print(f'📁 Drive backup dir: {DRIVE_CKPT_BACKUP}')

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 8: 🚀 START TRAINING  (Resume from step 7,000)
# ═══════════════════════════════════════════════════════
import os, subprocess, sys, glob

os.chdir(WORK_DIR)

# Confirm latest checkpoint before starting
ckpts = sorted(glob.glob(f'{CKPT_DIR}/model_ckpt_steps_*.ckpt'))
if ckpts:
    latest_step = int(ckpts[-1].split('steps_')[-1].replace('.ckpt',''))
    print(f'🔄 Resuming from step {latest_step:,} → target {TARGET_STEPS:,}')
    print(f'   Steps remaining: {TARGET_STEPS - latest_step:,}')
else:
    print('🆕 Starting fresh training')

print('\n🚀 Launching DiffSinger training...\n')
print('=' * 60)

# Run training — DiffSinger auto-resumes from latest checkpoint
!cd {WORK_DIR} && python scripts/train.py \
    --config usr/configs/vocalido.yaml \
    --exp_name {EXP_NAME}

print('\n' + '=' * 60)
print('✅ Training complete!')

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 9: Export Final Checkpoint to Drive
# ═══════════════════════════════════════════════════════
import glob, shutil, os, zipfile

ckpts = sorted(glob.glob(f'{CKPT_DIR}/model_ckpt_steps_*.ckpt'))
if not ckpts:
    print('❌ No checkpoints found!')
else:
    final_ckpt = ckpts[-1]
    step = int(final_ckpt.split('steps_')[-1].replace('.ckpt',''))
    print(f'📦 Final checkpoint: step {step:,}')

    # Copy to Drive
    export_dir = f'{DRIVE_BASE}/final_export'
    os.makedirs(export_dir, exist_ok=True)

    print('💾 Zipping checkpoint + config for export...')
    zip_path = f'{export_dir}/vocalido_final_step{step}.zip'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        zf.write(final_ckpt, os.path.basename(final_ckpt))
        for extra in ['config.yaml', 'spk_map.json', 'lang_map.json', 'dictionary-en.txt']:
            p = os.path.join(CKPT_DIR, extra)
            if os.path.exists(p):
                zf.write(p, extra)

    size_mb = os.path.getsize(zip_path) / 1e6
    print(f'✅ Exported: {os.path.basename(zip_path)} ({size_mb:.0f} MB)')
    print(f'📁 Location: {zip_path}')

    # Stop auto-save thread
    _stop_autosave.set()
    print('\n🏁 All done! Download the zip from your Google Drive.')

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 10: [OPTIONAL] ทดสอบเสียง inference จาก checkpoint
# ═══════════════════════════════════════════════════════
import glob, os

ckpts = sorted(glob.glob(f'{CKPT_DIR}/model_ckpt_steps_*.ckpt'))
if not ckpts:
    print('No checkpoint to test')
else:
    latest = ckpts[-1]
    step = int(latest.split('steps_')[-1].replace('.ckpt',''))
    print(f'🔊 Running inference test at step {step:,}...')

    test_ds = '''
- offset: 0.0
  text: "La la la la la"
  ph_seq: "La la la la la"
  ph_dur: "0.5 0.5 0.5 0.5 0.5"
  ph_num: "1 1 1 1 1"
  note_seq: "A4 A4 G4 E4 C4"
  note_dur: "0.5 0.5 0.5 0.5 0.5"
  note_slur: "0 0 0 0 0"
'''

    test_file = f'{WORK_DIR}/test_input.ds'
    with open(test_file, 'w') as f:
        f.write(test_ds)

    OUT_DIR = f'{WORK_DIR}/test_output'
    os.makedirs(OUT_DIR, exist_ok=True)

    !cd {WORK_DIR} && python scripts/infer.py acoustic \
        --config usr/configs/vocalido.yaml \
        --exp_name {EXP_NAME} \
        --ckpt_steps {step} \
        --ds {test_file} \
        --out_dir {OUT_DIR} \
        --title test_step_{step}

    wavs = glob.glob(f'{OUT_DIR}/*.wav')
    if wavs:
        print(f'✅ Test audio: {wavs[0]}')
        shutil.copy(wavs[0], f'{DRIVE_BASE}/test_step{step}.wav')
        print(f'💾 Saved to Drive: test_step{step}.wav')
    else:
        print('⚠️ No output wav found')